# Computational Patterns for Analog Deployment

This notebook provides a deep dive into the computational patterns across 7 neural network architecture families and their analog amenability for energy-efficient compute.

**For hardware architects:** Learn what computational patterns your hardware should support efficiently.

**For neural architects:** Learn which patterns to use for analog deployment and why.

**Goal:** Understand why different computational patterns have different analog tolerance and how to choose the right one for analog deployment.

## Why Computational Patterns Matter for Analog

Not all neural network computational structures are equally suited for analog hardware. The computational pattern determines:

- **Noise accumulation:** Single-pass patterns don't compound error; iterative/multi-step do
- **D/A boundary density:** Fewer ADC/DAC crossings = less energy and quantization error
- **Dynamics naturalness:** ODE/SDE patterns map directly to analog circuits
- **Precision requirements:** Some operations (softmax, layer norm) require digital precision

**For hardware architects:** Understanding these patterns helps you design hardware that supports the most analog-compatible computations efficiently.

**For neural architects:** Understanding these patterns helps you choose architectures that will work well on analog hardware.

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent
sys.path.insert(0, str(_ROOT))

import torch
import numpy as np

from neuro_analog.ir.types import ArchitectureFamily

## Overview of the 7 Architecture Families

In [ ]:
# List all supported architecture families
families = [
    (ArchitectureFamily.NEURAL_ODE, "Chen et al. 2018 - Continuous-time dynamics"),
    (ArchitectureFamily.SSM, "State Space Models (Mamba, S4) - Linear recurrence"),
    (ArchitectureFamily.DEQ, "Deep Equilibrium Models - Implicit fixed-point iteration"),
    (ArchitectureFamily.FLOW, "Normalizing Flows (FLUX) - Continuous velocity field"),
    (ArchitectureFamily.EBM, "Energy-Based Models - Hopfield dynamics"),
    (ArchitectureFamily.DIFFUSION, "Diffusion Models (Stable Diffusion) - Multi-step denoising"),
    (ArchitectureFamily.TRANSFORMER, "Attention-based models - FFN analog partition"),
]

print("Supported architecture families:")
for family, description in families:
    print(f"  {family.value:15s}: {description}")

## Computational Pattern Analysis

The key distinction is how computation flows through the network:

In [ ]:
computational_patterns = [
    ("Single-pass", ["Transformer", "Neural ODE", "SSM", "Flow", "EBM"], 
     "Noise doesn't accumulate - one forward pass"),
    ("Fixed-point iteration", ["DEQ"], 
     "Mismatch compounds across iterations until convergence"),
    ("Multi-step pipeline", ["Diffusion"], 
     "Quantization error accumulates across 100+ denoising steps"),
]

print("Computational patterns and noise behavior:")
for pattern, archs, behavior in computational_patterns:
    print(f"\n{pattern}:")
    print(f"  Architectures: {', '.join(archs)}")
    print(f"  Noise behavior: {behavior}")

## Analog Amenability Ranking

Based on cross-architecture tolerance studies, here's the amenability ranking:

In [ ]:
amenability_ranking = [
    ("Neural ODE", "Highest", "ODE form maps directly to RC integrators - natural analog paradigm for hardware"),
    ("SSM (Mamba)", "High", "Linear recurrence, minimal D/A boundaries, diagonal A structure - maps to parallel RC circuits"),
    ("EBM", "High", "Energy landscape wide - noise just blurs attractor basins, robust to perturbations"),
    ("Flow (FLUX)", "Medium", "Many function evaluations accumulate error, but ODE form helps - velocity field maps to RC integrators"),
    ("Transformer", "Medium", "FFN analog partition works, but softmax/layer norm digital-required - precision-critical operations"),
    ("DEQ", "Low", "Fixed-point iteration compounds mismatch - feedback circuits natural but convergence sensitive"),
    ("Diffusion", "Lowest", "100+ denoising steps - each adds quantization error, error accumulation across pipeline"),
]

print("Analog amenability ranking (from cross-arch sweep):")
for i, (name, level, reason) in enumerate(amenability_ranking, 1):
    print(f"{i}. {name:20s} ({level:8s}): {reason}")

## Physical Explanations for Ranking

### Why Neural ODE is highest amenability

- **Natural ODE form:** dx/dt = f_θ(x,t) IS the input format for Ark (analog circuit compiler)
- **RC integrator mapping:** ODE solver maps directly to op-amp RC integrator circuits
- **Single forward pass:** No iterative error accumulation
- **Few D/A boundaries:** Continuous-time dynamics minimize ADC/DAC crossings

### Why SSM (Mamba) is high amenability

- **Diagonal A matrix:** SSMs use diagonal state matrices, which map to parallel RC circuits
- **Linear recurrence:** Simple x_t = A x_{t-1} + B u_t structure
- **Minimal boundaries:** State stays in analog domain across sequence
- **Decay structure:** A_re decay helps stabilize under mismatch

### Why EBM is high amenability

- **Hopfield dynamics:** Energy minimization maps to analog feedback circuits
- **Wide attractor basins:** Noise just blurs basins, doesn't destroy them
- **Thermodynamic sampling:** Can use physical thermal noise as computational resource
- **Robust to perturbations:** Energy landscape provides natural regularization

### Why Flow is medium amenability

- **ODE form helps:** Velocity field ODE maps to RC integrators
- **Many NFE:** Number of function evaluations accumulates error
- **Lipschitz constraints:** Required for invertibility, may conflict with analog nonidealities
- **Better than diffusion:** Fewer steps than diffusion, but more than single-pass

### Why Transformer is medium amenability

- **FFN analog partition:** Feed-forward networks can be analogized
- **Softmax digital-required:** Global comparison requires high precision
- **Layer norm digital-required:** Mean/variance computation needs digital accuracy
- **Attention complexity:** Q·K^T is data-dependent, harder to map to static crossbars

### Why DEQ is low amenability

- **Fixed-point iteration:** z* = f_θ(z*) requires repeated application
- **Mismatch compounds:** Each iteration adds error, may never converge
- **Spectral radius:** Under mismatch, ρ(f') may exceed 1 → divergence
- **Natural analog but fragile:** Feedback circuits are natural but convergence is sensitive

### Why Diffusion is lowest amenability

- **100+ denoising steps:** Each step adds quantization error
- **Error accumulation:** Small per-step error compounds to large total error
- **Complex scheduler:** β(t) schedule requires precise timing
- **U-Net structure:** Skip connections create many D/A boundaries

## Circuit Mode Mappings

Different architecture families use different analog circuit primitives:

In [ ]:
circuit_modes = [
    ("rc_integrator", ["Neural ODE", "Flow", "SSM"], 
     "RC circuit ODE solver with Johnson-Nyquist noise (kT/C)"),
    ("hopfield", ["DEQ", "EBM"], 
     "Continuous-time analog feedback relaxation to fixed point"),
    ("classic", ["Diffusion"], 
     "DDIM (deterministic) diffusion - discrete time steps"),
    ("cld", ["Diffusion"], 
     "Critically-damped Langevin dynamics - continuous-time SDE"),
]

print("Circuit mode mappings:")
for mode, archs, description in circuit_modes:
    print(f"\n{mode}:")
    print(f"  Architectures: {', '.join(archs)}")
    print(f"  Physical implementation: {description}")

## Energy/Latency Estimates per Family

Using HardwareProfile, we can estimate energy and latency for each architecture family:

In [ ]:
from neuro_analog.ir.energy_model import HardwareProfile

# Load default hardware profile
profile = HardwareProfile()

print("Hardware profile constants (sourced from IBM PCM, SRAM IMC, AIMC surveys):")
print(f"  Analog MAC energy: {profile.analog_mac_energy_pJ} pJ")
print(f"  Digital MAC energy: {profile.digital_mac_energy_pJ} pJ")
print(f"  ADC energy: {profile.adc_energy_pJ} pJ")
print(f"  Integrator energy: {profile.integrator_energy_pJ_per_state} pJ/state")
print(f"\nEnergy savings: {profile.digital_mac_energy_pJ / profile.analog_mac_energy_pJ:.1f}x")

## When to Choose Which Computational Pattern

**Choose Neural ODE pattern if:**
- You need the highest analog amenability
- Your problem is naturally continuous-time (time series, control)
- You want to use Ark for circuit compilation
- Hardware: Your substrate supports RC integrators well

**Choose SSM (Mamba) pattern if:**
- You need long-range dependencies with linear complexity
- You want high amenability with sequence modeling
- Diagonal structure fits your use case
- Hardware: Your substrate supports parallel RC circuits

**Choose EBM pattern if:**
- You're doing generative modeling with energy landscapes
- You can leverage thermodynamic sampling
- Robustness to noise is acceptable
- Hardware: Your substrate supports feedback relaxation

**Choose Flow pattern if:**
- You need exact likelihood estimation
- ODE form is acceptable for your application
- You can tolerate medium amenability
- Hardware: Your substrate supports RC integrators for velocity fields

**Choose Transformer pattern if:**
- You need strong attention mechanisms
- You're okay with digital-required softmax/layer norm
- FFN-only analog partition is sufficient
- Hardware: Your substrate supports crossbar arrays for FFN layers

**Choose DEQ pattern if:**
- You need implicit depth without unrolling
- You can add calibration/stabilization for mismatch
- Fixed-point convergence is critical
- Hardware: Your substrate supports feedback circuits with convergence guarantees

**Choose Diffusion pattern if:**
- You need state-of-the-art image generation
- You can accept low analog amenability (mostly digital)
- Quality is more important than energy efficiency
- Hardware: You have digital fallback for precision-critical operations

## Links to Architecture-Specific Examples

For detailed workflows for each architecture family, see:

- **EBM:** `examples/06_ebm_ark.py`
- **Flow:** `examples/07_flow_ark.py`
- **Diffusion:** `examples/08_diffusion_ark.py`
- **Transformer FFN:** `examples/09_transformer_ffn_ark.py`
- **DEQ:** `examples/10_deq_ark.py`
- **SSM:** `examples/11_ssm_ark.py`
- **Neural ODE:** `examples/03_ark_pipeline.py` (full Ark workflow)

## Key Takeaways

1. **Computational structure determines analog tolerance** - Single-pass > iterative > multi-step
2. **Neural ODE has highest amenability** - Natural ODE form maps to RC integrators
3. **Diffusion has lowest amenability** - 100+ steps accumulate quantization error
4. **Circuit modes map to physics** - rc_integrator, hopfield, cld have real analog implementations
5. **Energy savings are consistent** - 20x improvement regardless of architecture (for analog portions)
6. **Choose architecture based on amenability + task fit** - Don't sacrifice task quality for energy